In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

# -----------------------------
# 1. Load IMDb Dataset
# -----------------------------
data = load_dataset("imdb")

train = data["train"]
test = data["test"]

# Use a smaller dataset for faster college execution
train_texts = train["text"][:10000]
train_labels = train["label"][:10000]

test_texts = test["text"][:2500]
test_labels = test["label"][:2500]

# -----------------------------
# 2. Create Vocabulary
# -----------------------------
words = []

for text in train_texts:
    words.extend(text.lower().split())

counter = Counter(words)

VOCAB_SIZE = 10000
word_to_id = {"<PAD>": 0, "<UNK>": 1}

for word, _ in counter.most_common(VOCAB_SIZE - 2):
    word_to_id[word] = len(word_to_id)

# -----------------------------
# 3. Convert Text to Numbers
# -----------------------------
MAX_LEN = 200

def encode(text):
    ids = [
        word_to_id.get(word, 1)
        for word in text.lower().split()
    ]

    ids = ids[:MAX_LEN]
    ids += [0] * (MAX_LEN - len(ids))

    return ids

X_train = torch.tensor([encode(x) for x in train_texts])
y_train = torch.tensor(train_labels, dtype=torch.float32)

X_test = torch.tensor([encode(x) for x in test_texts])
y_test = torch.tensor(test_labels, dtype=torch.float32)

# -----------------------------
# 4. DataLoader
# -----------------------------
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

# -----------------------------
# 5. LSTM Model
# -----------------------------
class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(word_to_id), 128
        )

        self.lstm = nn.LSTM(
            128, 64, batch_first=True
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        return self.fc(hidden[-1]).squeeze(1)


model = LSTMModel()

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# -----------------------------
# 6. Train
# -----------------------------
for epoch in range(3):

    model.train()
    total_loss = 0

    for x, y in train_loader:

        optimizer.zero_grad()

        output = model(x)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/3 - "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

# -----------------------------
# 7. Evaluation
# -----------------------------
model.eval()

predictions = []

with torch.no_grad():

    for x, y in test_loader:

        output = torch.sigmoid(model(x))

        pred = (output >= 0.5).int()

        predictions.extend(pred.numpy())

print("\n--- Performance ---")

print("Accuracy :", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall   :", recall_score(y_test, predictions))
print("F1 Score :", f1_score(y_test, predictions))

# -----------------------------
# 8. New Review Prediction
# -----------------------------
def predict_review(review):

    x = torch.tensor([encode(review)])

    with torch.no_grad():

        probability = torch.sigmoid(
            model(x)
        ).item()

    sentiment = (
        "Positive"
        if probability >= 0.5
        else "Negative"
    )

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print(f"Confidence: {probability:.2%}")


predict_review(
    "This movie was amazing and I really enjoyed it."
)

ModuleNotFoundError: No module named 'datasets'

In [2]:
import os
import zipfile
import urllib.request
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

# =========================
# 1. Download IMDb Dataset
# =========================

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
file = "aclImdb_v1.tar.gz"

if not os.path.exists(file):
    print("Downloading IMDb dataset...")
    urllib.request.urlretrieve(url, file)

print("Dataset downloaded.")

# =========================
# 2. Extract Dataset
# =========================

import tarfile

if not os.path.exists("aclImdb"):
    print("Extracting...")
    with tarfile.open(file, "r:gz") as tar:
        tar.extractall(".")

print("Dataset ready.")

# =========================
# 3. Read Reviews
# =========================

def read_reviews(folder, limit):
    texts = []
    labels = []

    for label, value in [("pos", 1), ("neg", 0)]:
        path = os.path.join(folder, label)

        for filename in os.listdir(path)[:limit]:
            if filename.endswith(".txt"):
                with open(
                    os.path.join(path, filename),
                    encoding="utf-8"
                ) as f:
                    texts.append(f.read())
                    labels.append(value)

    return texts, labels


train_texts, train_labels = read_reviews(
    "aclImdb/train", 5000
)

test_texts, test_labels = read_reviews(
    "aclImdb/test", 1000
)

print("Training reviews:", len(train_texts))
print("Testing reviews :", len(test_texts))

# =========================
# 4. Create Vocabulary
# =========================

counter = Counter()

for text in train_texts:
    counter.update(text.lower().split())

VOCAB_SIZE = 10000

word_to_id = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, _ in counter.most_common(VOCAB_SIZE - 2):
    word_to_id[word] = len(word_to_id)

# =========================
# 5. Encode Reviews
# =========================

MAX_LEN = 200

def encode(text):
    words = text.lower().split()

    ids = [
        word_to_id.get(word, 1)
        for word in words[:MAX_LEN]
    ]

    ids += [0] * (MAX_LEN - len(ids))

    return ids


X_train = torch.tensor(
    [encode(x) for x in train_texts],
    dtype=torch.long
)

y_train = torch.tensor(
    train_labels,
    dtype=torch.float32
)

X_test = torch.tensor(
    [encode(x) for x in test_texts],
    dtype=torch.long
)

y_test = torch.tensor(
    test_labels,
    dtype=torch.float32
)

# =========================
# 6. DataLoader
# =========================

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

# =========================
# 7. LSTM Model
# =========================

class SentimentLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(word_to_id), 128
        )

        self.lstm = nn.LSTM(
            128,
            64,
            batch_first=True
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x):

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        return self.fc(
            hidden[-1]
        ).squeeze(1)


model = SentimentLSTM()

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# =========================
# 8. Training
# =========================

for epoch in range(3):

    model.train()
    total_loss = 0

    for x, y in train_loader:

        optimizer.zero_grad()

        output = model(x)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/3 | "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

# =========================
# 9. Evaluation
# =========================

model.eval()

predictions = []

with torch.no_grad():

    for x, y in test_loader:

        output = torch.sigmoid(model(x))

        pred = (output >= 0.5).int()

        predictions.extend(
            pred.numpy()
        )

print("\n===== MODEL PERFORMANCE =====")

print(
    "Accuracy :",
    accuracy_score(y_test, predictions)
)

print(
    "Precision:",
    precision_score(y_test, predictions)
)

print(
    "Recall   :",
    recall_score(y_test, predictions)
)

print(
    "F1 Score :",
    f1_score(y_test, predictions)
)

# =========================
# 10. New Review
# =========================

def predict_review(review):

    x = torch.tensor(
        [encode(review)],
        dtype=torch.long
    )

    with torch.no_grad():

        probability = torch.sigmoid(
            model(x)
        ).item()

    if probability >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print(
        f"Confidence: {probability:.2%}"
    )


predict_review(
    "This movie was excellent and amazing."
)

Dataset downloaded.
Extracting...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_31072\1673527916.py:32: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(".")


Dataset ready.
Training reviews: 10000
Testing reviews : 2000
Epoch 1/3 | Loss: 0.6940
Epoch 2/3 | Loss: 0.6848
Epoch 3/3 | Loss: 0.6639

===== MODEL PERFORMANCE =====
Accuracy : 0.512
Precision: 0.5342857142857143
Recall   : 0.187
F1 Score : 0.277037037037037

Review: This movie was excellent and amazing.
Sentiment: Negative
Confidence: 49.91%


In [3]:
import os
import re
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter

# -----------------------------
# 1. Read Dataset
# -----------------------------

def read_reviews(folder, limit):
    texts, labels = [], []

    for label, value in [("pos", 1), ("neg", 0)]:
        path = os.path.join(folder, label)
        files = [f for f in os.listdir(path) if f.endswith(".txt")]

        for filename in files[:limit]:
            with open(os.path.join(path, filename),
                      encoding="utf-8") as f:
                texts.append(f.read())
                labels.append(value)

    return texts, labels


train_texts, train_labels = read_reviews(
    "aclImdb/train", 5000
)

test_texts, test_labels = read_reviews(
    "aclImdb/test", 1000
)

print("Training:", len(train_texts))
print("Testing :", len(test_texts))

# -----------------------------
# 2. Clean Text
# -----------------------------

def clean(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return text.split()


# -----------------------------
# 3. Vocabulary
# -----------------------------

counter = Counter()

for text in train_texts:
    counter.update(clean(text))

VOCAB_SIZE = 12000

word_to_id = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, _ in counter.most_common(VOCAB_SIZE - 2):
    word_to_id[word] = len(word_to_id)

print("Vocabulary:", len(word_to_id))

# -----------------------------
# 4. Encode
# -----------------------------

MAX_LEN = 200

def encode(text):
    words = clean(text)

    ids = [
        word_to_id.get(w, 1)
        for w in words[:MAX_LEN]
    ]

    ids += [0] * (MAX_LEN - len(ids))

    return ids


X_train = torch.tensor(
    [encode(x) for x in train_texts],
    dtype=torch.long
)

y_train = torch.tensor(
    train_labels,
    dtype=torch.float32
)

X_test = torch.tensor(
    [encode(x) for x in test_texts],
    dtype=torch.long
)

y_test = torch.tensor(
    test_labels,
    dtype=torch.float32
)

# -----------------------------
# 5. DataLoader
# -----------------------------

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

# -----------------------------
# 6. BiLSTM Model
# -----------------------------

class SentimentLSTM(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(word_to_id),
            128,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            128,
            64,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(128, 1)

    def forward(self, x):

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        hidden = torch.cat(
            (hidden[-2], hidden[-1]),
            dim=1
        )

        hidden = self.dropout(hidden)

        return self.fc(hidden).squeeze(1)


model = SentimentLSTM()

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# -----------------------------
# 7. Training
# -----------------------------

EPOCHS = 5

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for x, y in train_loader:

        optimizer.zero_grad()

        output = model(x)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

# -----------------------------
# 8. Evaluation
# -----------------------------

model.eval()
predictions = []

with torch.no_grad():

    for x, y in test_loader:

        output = torch.sigmoid(model(x))

        pred = (output >= 0.5).int()

        predictions.extend(pred.numpy())


print("\n===== MODEL PERFORMANCE =====")

print(
    f"Accuracy : {accuracy_score(y_test, predictions):.4f}"
)

print(
    f"Precision: {precision_score(y_test, predictions):.4f}"
)

print(
    f"Recall   : {recall_score(y_test, predictions):.4f}"
)

print(
    f"F1 Score : {f1_score(y_test, predictions):.4f}"
)

# -----------------------------
# 9. New Review Prediction
# -----------------------------

def predict_review(review):

    x = torch.tensor(
        [encode(review)],
        dtype=torch.long
    )

    model.eval()

    with torch.no_grad():

        probability = torch.sigmoid(
            model(x)
        ).item()

    sentiment = (
        "Positive"
        if probability >= 0.5
        else "Negative"
    )

    print("\nReview:", review)
    print("Sentiment:", sentiment)
    print(f"Confidence: {probability:.2%}")


predict_review(
    "This movie was excellent and amazing. "
    "I really enjoyed watching it."
)

Training: 10000
Testing : 2000
Vocabulary: 12000
Epoch 1/5 | Loss: 0.6816
Epoch 2/5 | Loss: 0.5913
Epoch 3/5 | Loss: 0.4642
Epoch 4/5 | Loss: 0.3457
Epoch 5/5 | Loss: 0.2693

===== MODEL PERFORMANCE =====
Accuracy : 0.7685
Precision: 0.7915
Recall   : 0.7290
F1 Score : 0.7590

Review: This movie was excellent and amazing. I really enjoyed watching it.
Sentiment: Positive
Confidence: 59.47%
